# Notebook 02 — Datenaufbereitung

**SBB Tracker · ZHAW Scientific Programming FS2026**
Joël Hasler & Patrick Ferreira

Aufbauend auf der Datenbank aus Notebook 01 bereiten wir die Rohdaten
für die statistische Analyse vor. Konkret:

1. **Datenqualität checken** — Missings, Duplikate, Outlier
2. **Filter auf saubere Daten** — nur `an_prognose_status == 'REAL'`
3. **Zeit-Features ableiten** — Stunde, Wochentag, Rush-Hour-Flag
4. **Stationen mit Wetterstationen verknüpfen** — via KDTree-Nearest
5. **Wetterdaten joinen** — pro Verspätungs-Event Temperatur, Regen, Wind
6. **Verspätungen klassifizieren** — kategoriale Buckets (pünktlich,
   leicht, klassisch, stark, extrem)
7. **Speichern** als Parquet für Notebook 03

Hilfsfunktionen sind in `app/utils.py` ausgelagert (OOP-Bonus + Wiederverwendung
in der Streamlit-Webapp).


## Bibliotheken und Einstellungen


In [1]:
# Bibliotheken und Einstellungen
import os
import sqlite3
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(color_codes=True)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print("Aktuelles Verzeichnis:", os.getcwd())
print("Projekt-Root:", PROJECT_ROOT)


Aktuelles Verzeichnis: C:\Users\hasle\Documents\Scientific Programming\SBB Tracker\project\notebooks
Projekt-Root: C:\Users\hasle\Documents\Scientific Programming\SBB Tracker\project


In [2]:
# Utility-Modul aus app/ einbinden
import sys
sys.path.insert(0, str(PROJECT_ROOT / "app"))
import utils
print("utils.py geladen aus:", PROJECT_ROOT / "app" / "utils.py")
print("DB-Pfad:", utils.db_path())


utils.py geladen aus: C:\Users\hasle\Documents\Scientific Programming\SBB Tracker\project\app\utils.py
DB-Pfad: C:\Users\hasle\Documents\Scientific Programming\SBB Tracker\project\data\processed\sbb_tracker.db


## Daten aus der Datenbank laden

Wir lesen die in Notebook 01 erstellten Tabellen direkt mit
`pd.read_sql(...)`. Indizes auf `bpuic` und `betriebstag` (in NB01
angelegt) beschleunigen die Joins.


In [3]:
df_stations = utils.query("SELECT * FROM stations")
df_weather = utils.query("SELECT * FROM weather_hourly")
df_delays = utils.query("SELECT * FROM delays")
print(f"Stationen:  {len(df_stations):>9,} Zeilen")
print(f"Wetter:     {len(df_weather):>9,} Zeilen")
print(f"Verspaetung:{len(df_delays):>9,} Zeilen")


Stationen:      1,743 Zeilen
Wetter:        50,040 Zeilen
Verspaetung:3,200,604 Zeilen


## Datenqualitäts-Check

Erst pro Tabelle: wie viele Missing-Werte? Gibt es Duplikate? Wie
verteilen sich die Zeitstempel? Das ist Standard-EDA, gibt aber
Hinweise auf Cleaning-Bedarf bevor wir Statistik darauf rechnen.


In [4]:
def quality_report(name, df):
    print(f"=== {name} ({len(df):,} Zeilen) ===")
    missing = df.isna().sum()
    missing_pct = (100 * missing / len(df)).round(2)
    report = pd.DataFrame({"missing": missing, "pct": missing_pct})
    report = report[report["missing"] > 0].sort_values("missing", ascending=False)
    if len(report) == 0:
        print("  Keine Missings.")
    else:
        print(report.head(10).to_string())
    print(f"  Duplikate (exakt): {df.duplicated().sum():,}")
    print()

quality_report("Stationen", df_stations)
quality_report("Wetter", df_weather)
quality_report("Verspaetungen", df_delays)


=== Stationen (1,743 Zeilen) ===
                    missing   pct
cantonabbreviation       33  1.89
cantonname               33  1.89
abbreviation             16  0.92
  Duplikate (exakt): 0

=== Wetter (50,040 Zeilen) ===
  Keine Missings.
  Duplikate (exakt): 0

=== Verspaetungen (3,200,604 Zeilen) ===
                     missing    pct
ab_prognose           391838  12.24
delay_dep_sec         391838  12.24
an_prognose           391689  12.24
delay_arr_sec         391689  12.24
abfahrtszeit          265132   8.28
ankunftszeit          264826   8.27
an_prognose_status    253447   7.92
ab_prognose_status    252286   7.88
verkehrsmittel_text        8   0.00
linien_text                8   0.00


  Duplikate (exakt): 0



## Auf saubere Messungen filtern (`AN_PROGNOSE_STATUS == 'REAL'`)

Die Spalte `an_prognose_status` unterscheidet zwischen `REAL` (echte
gemessene Ankunftszeit) und `PROGNOSE` (geschätzt — z.B. weil der Zug
noch nicht angekommen ist oder die Messung nicht funktionierte). Für
seriöse Statistik verwenden wir nur `REAL`.


In [5]:
n_before = len(df_delays)
df = df_delays.loc[df_delays["an_prognose_status"] == "REAL"].copy()
df = df.loc[df["delay_arr_sec"].notna()].copy()
n_after = len(df)
print(f"Vorher:  {n_before:,} Records")
print(f"Nachher: {n_after:,} Records  ({100*n_after/n_before:.1f}%)")


Vorher:  3,200,604 Records
Nachher: 2,739,804 Records  (85.6%)


### Ausreisser inspizieren

Verspätungen mit > 30 Minuten sind ungewöhnlich — meist Folge von
Notfällen, Streckensperrungen oder fehlerhaften Messungen. Wir
behalten sie, weisen sie aber explizit aus.


In [6]:
print("Verteilung Ankunftsverspaetung (Sekunden):")
print(df["delay_arr_sec"].describe(percentiles=[.5, .9, .95, .99, .999]).to_string())
n_extreme = (df["delay_arr_sec"] > 1800).sum()
print(f"\nExtreme Verspaetungen (>30 Min): {n_extreme:,} "
      f"({100*n_extreme/len(df):.3f}% aller Faelle)")


Verteilung Ankunftsverspaetung (Sekunden):
count    2.739804e+06
mean     4.544584e+01
std      2.020593e+02
min     -1.730920e+05
50%      2.800000e+01
90%      1.270000e+02
95%      1.810000e+02
99%      4.090000e+02
99.9%    1.483000e+03
max      1.462000e+04

Extreme Verspaetungen (>30 Min): 1,921 (0.070% aller Faelle)


## Zeit-Features ableiten

Aus dem Zeitstempel `ankunftszeit` extrahieren wir Stunde, Wochentag,
Wochenende-Flag und Rush-Hour-Flag. Diese Features werden in den
Stats-Tests von Notebook 03 als Gruppierungs-Variablen benutzt
(Werktag vs. Wochenende, Stunde vs. Verspätung).


In [7]:
df = utils.add_time_features(df, ts_col="ankunftszeit")
cols_to_show = ["betriebstag", "haltestellen_name", "ankunftszeit",
                "delay_arr_sec", "hour", "weekday", "is_weekend", "is_rush_hour"]
df[cols_to_show].head(5)


,betriebstag,haltestellen_name,ankunftszeit,delay_arr_sec,hour,weekday,is_weekend,is_rush_hour
0,2026-03-31 00:00:00,Chiasso,2026-03-31 07:58:00,-78.0,7,Di,False,True
1,2026-03-31 00:00:00,Lugano,2026-03-31 08:28:00,-9.0,8,Di,False,True
2,2026-03-31 00:00:00,Bellinzona,2026-03-31 08:46:00,-28.0,8,Di,False,True
3,2026-03-31 00:00:00,Arth-Goldau,2026-03-31 09:42:00,2.0,9,Di,False,False
4,2026-03-31 00:00:00,Zug,2026-03-31 10:00:00,175.0,10,Di,False,False


In [8]:
# Verteilung der Faelle ueber Stunde + Wochentag
crosstab = pd.crosstab(df["weekday"], df["hour"])
crosstab = crosstab.reindex(["Mo", "Di", "Mi", "Do", "Fr", "Sa", "So"])
crosstab


hour,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
weekday,,,,,,,,,,,,,,,,,,,,,,,,
Mo,10479,1874,633,565,2064,13432,21484,22489,21723,20881,20779,20872,20915,20819,20723,21039,21695,22356,22071,21348,20450,18943,17356,16241
Di,10741,1735,275,276,2011,13875,21966,23000,21942,21004,20861,20939,20986,20896,20837,21164,21919,22758,22362,21550,20590,19066,17568,16471
Mi,10818,1720,230,236,1711,11879,18725,19642,18803,18031,17877,17930,17990,17952,17831,18077,18690,19435,19184,18441,17574,16333,15088,14196
Do,9508,2034,711,644,2128,13393,21386,22484,21663,20813,20667,20764,20765,20654,20617,20916,21518,22204,21943,21309,20389,18920,17491,16544
Fr,11189,2934,1376,1204,2423,13309,21218,22216,21492,20714,20578,20670,20685,20608,20525,20786,21396,21999,21723,21094,20313,19410,18280,17392
Sa,12658,5689,3754,3112,2941,11249,19111,20185,20444,20276,20149,20331,20197,20071,20011,20209,20343,20369,20363,20269,19532,19012,18167,17238
So,12364,5505,3659,3081,2929,10784,18604,19647,19916,19834,19758,19882,19860,19726,19735,19915,20082,20049,20002,19967,19548,18489,17201,16172


## Bahnhöfe ↔ Wetterstationen mappen

Pro Bahnhof finden wir die geografisch nächste MeteoSchweiz-Station
(von 15 Stationen) via scipy KDTree (Euklidische Distanz auf lat/lon).
Das ist deutlich schneller als ein Haversine-Loop und für die Schweiz
ausreichend präzise.


In [9]:
df_stations_mapped = utils.map_stations_to_weather(df_stations)
print("Beispiel-Mapping (10 Bahnhoefe):")
print(df_stations_mapped[["designationofficial", "cantonabbreviation",
                          "nearest_weather_abbr", "weather_distance_km"]]
      .sample(10, random_state=42).to_string(index=False))


Beispiel-Mapping (10 Bahnhoefe):
designationofficial cantonabbreviation nearest_weather_abbr  weather_distance_km
          Dietlikon                 ZH                  SMA             7.516747
            Pensier                 FR                  NEU            26.780174
 Villette (Vigezzo)                NaN                  MAG            45.910431
        Haldenstein                 GR                  CHU             0.806465
          Steinmaur                 ZH                  KLO             9.956689
          Benken SG                 SG                  STG            49.840079
       Vendlincourt                 JU                  BAS            48.382523
      Sendy-Sollard                 VD                  SIO            51.187521
          Vauderens                 FR                  NEU            43.553769
        Les Fumeaux                 VS                  SIO            31.916712


In [10]:
# Verteilung: wie viele Bahnhoefe pro Wetterstation?
mapping_dist = (df_stations_mapped["nearest_weather_abbr"]
                .value_counts()
                .sort_values(ascending=False))
print("Bahnhoefe pro Wetterstation:")
print(mapping_dist.to_string())


Bahnhoefe pro Wetterstation:
nearest_weather_abbr
NEU    236
SIO    197
SMA    190
BER    171
STG    165
WYN    125
KLO    116
LUZ    115
GVE     96
INT     80
DAV     68
MAG     66
CHU     57
BAS     35
LUG     26


## Wetterdaten an Verspätungs-Events joinen

Für jeden Verspätungs-Event holen wir die Wetterdaten der nächsten
Wetterstation zur passenden Stunde. Strategie:

1. Verspätungs-DF mit `nearest_weather_abbr` per BPUIC-Join anreichern
2. Wetter-Zeitstempel auf Stunde runden (stündliche Wetterdaten)
3. Merge auf `(weather_abbr, weather_hour)`

Das ergibt für jeden Halt: Temperatur, Niederschlag, Wind, Feuchtigkeit
in der Stunde des Halts.


In [11]:
# Schritt 1: nearest_weather_abbr an delays anhaengen
station_lookup = df_stations_mapped.set_index("number")["nearest_weather_abbr"]
df["weather_abbr"] = df["bpuic"].map(station_lookup)
n_matched = df["weather_abbr"].notna().sum()
print(f"Verspaetungen mit Wetter-Mapping: {n_matched:,}/{len(df):,} "
      f"({100*n_matched/len(df):.1f}%)")


Verspaetungen mit Wetter-Mapping: 2,739,800/2,739,804 (100.0%)


In [12]:
# Schritt 2: Wetter-Zeitstempel auf Stunde runden + delay-Zeitstempel auch
df_weather["weather_ts_hour"] = (pd.to_datetime(df_weather["reference_timestamp"])
                                 .dt.floor("h"))
df["delay_ts_hour"] = pd.to_datetime(df["ankunftszeit"]).dt.floor("h")

# Schritt 3: Merge
df_with_weather = df.merge(
    df_weather[["station_abbr", "weather_ts_hour",
                "tre200h0", "rre150h0", "fkl010h0", "ure200h0", "sre000h0"]],
    left_on=["weather_abbr", "delay_ts_hour"],
    right_on=["station_abbr", "weather_ts_hour"],
    how="left",
)

# Aufraeumen: Hilfs-Spalten droppen, sinnvolle Namen vergeben
df_with_weather = df_with_weather.drop(columns=["station_abbr", "weather_ts_hour"])
df_with_weather = df_with_weather.rename(columns={
    "tre200h0": "temperatur_c",
    "rre150h0": "niederschlag_mm",
    "fkl010h0": "wind_ms",
    "ure200h0": "feuchte_pct",
    "sre000h0": "sonne_min",
})
n_weather = df_with_weather["temperatur_c"].notna().sum()
print(f"Records mit Wetterdaten: {n_weather:,}/{len(df_with_weather):,} "
      f"({100*n_weather/len(df_with_weather):.1f}%)")
df_with_weather[["haltestellen_name", "ankunftszeit", "delay_arr_sec",
                 "temperatur_c", "niederschlag_mm", "wind_ms"]].head(5)


Records mit Wetterdaten: 2,738,031/2,739,804 (99.9%)


,haltestellen_name,ankunftszeit,delay_arr_sec,temperatur_c,niederschlag_mm,wind_ms
0,Chiasso,2026-03-31 07:58:00,-78.0,8.8,0.0,4.6
1,Lugano,2026-03-31 08:28:00,-9.0,9.5,0.0,5.2
2,Bellinzona,2026-03-31 08:46:00,-28.0,8.1,0.0,1.5
3,Arth-Goldau,2026-03-31 09:42:00,2.0,2.7,0.4,2.2
4,Zug,2026-03-31 10:00:00,175.0,4.0,0.0,3.3


## Verspätungs-Klassifikation

Aus der Sekunden-Spalte `delay_arr_sec` leiten wir eine kategoriale
Spalte `delay_class` ab — von "frueh_30+s" bis "extrem_ueber_10min".
Das gibt uns gleichzeitig die "klassische Verspätung" (>3 Min, SBB-
offizielle Definition) als binäres Feature.


In [13]:
df_final = utils.add_delay_class(df_with_weather, col="delay_arr_sec")
df_final["is_late_3min"] = df_final["delay_arr_sec"] > 180

# Verteilung der Klassen
class_dist = df_final["delay_class"].value_counts().sort_index()
print("Verspaetungs-Klassen:")
print(class_dist.to_string())
print(f"\nKlassisch verspaetet (>3 Min): {df_final['is_late_3min'].sum():,} "
      f"({100*df_final['is_late_3min'].mean():.2f}%)")


Verspaetungs-Klassen:
delay_class
extrem_ueber_10min         14866
frueh_30+s                310265
frueh_unter_30s           459330
leicht_1_3min             663323
puenktlich_unter_1min    1167496
stark_5_10min              33072
verspaetet_3_5min          91452

Klassisch verspaetet (>3 Min): 137,798 (5.03%)


## Speichern als `delays_prepared.parquet`

Notebook 03 nimmt diese Datei als Input. Wir speichern als Parquet
(komprimiert, typisiert) statt CSV — pandas liest's 10× schneller.


In [14]:
OUT = DATA_PROCESSED / "delays_prepared.parquet"
df_final.to_parquet(OUT, compression="snappy", index=False)
print(f"Gespeichert: {OUT}")
print(f"Groesse:     {OUT.stat().st_size / 1024**2:.1f} MB")
print(f"Zeilen:      {len(df_final):,}")
print(f"Spalten:     {df_final.shape[1]}")


Gespeichert: C:\Users\hasle\Documents\Scientific Programming\SBB Tracker\project\data\processed\delays_prepared.parquet
Groesse:     62.2 MB
Zeilen:      2,739,804
Spalten:     31


## Zusammenfassung Notebook 02

Aus den drei Rohtabellen ist nun ein einziger angereicherter Datensatz
entstanden. Pro Verspätungs-Event haben wir:
- **Zeitliche Features**: Stunde, Wochentag, Rush-Hour-Flag
- **Wetter-Features**: Temperatur, Regen, Wind, Feuchtigkeit, Sonne
- **Geografisches**: Kanton, lat/lon (über Stations-JOIN)
- **Klassifikation**: Bucket-Label + binäres `is_late_3min`

Notebook 03 testet damit vier Hypothesen:
1. Werktag vs. Wochenende (t-Test / Mann-Whitney)
2. Zugtyp (S-Bahn vs. IC) — ANOVA
3. Wetter ↔ Verspätung — Pearson/Spearman-Korrelation mit p-value
4. Multivariate OLS-Regression als Krönung


In [15]:
# System-Info (Reproduzierbarkeits-Footer)
import platform
from platform import python_version
from datetime import datetime

print("-----------------------------------")
print(os.name.upper())
print(platform.system(), "|", platform.release())
print("Datetime:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("Python Version:", python_version())
print("-----------------------------------")


-----------------------------------
NT
Windows | 11
Datetime: 2026-05-20 23:47:21
Python Version: 3.12.10
-----------------------------------
